In [ ]:
"""
Mini Pipeline de Análise de Dados - Vendas de uma Loja
--------------------------------------------------------
Objetivo: praticar um fluxo simples de ETL (Extract, Transform, Load)
usando Python e Pandas, simulando um cenário real de análise de dados.

Etapas do pipeline:
1. EXTRACT  -> criar/ler os dados brutos
2. TRANSFORM -> limpar, tratar nulos, criar colunas novas, agrupar
3. LOAD     -> salvar o resultado tratado em um novo arquivo CSV
"""

import pandas as pd
import numpy as np


# 1. EXTRACT — criando um dataset fictício (simulando dados "sujos")

dados_brutos = {
    "id_venda": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "produto": ["Notebook", "Mouse", "Teclado", "Monitor", "Notebook",
                "Mouse", "Cadeira", "Monitor", "Teclado", np.nan],
    "categoria": ["Eletrônicos", "Acessórios", "Acessórios", "Eletrônicos",
                  "Eletrônicos", "Acessórios", "Móveis", "Eletrônicos",
                  "Acessórios", "Acessórios"],
    "preco_unitario": [3500.00, 50.00, 120.00, 900.00, 3500.00,
                       np.nan, 450.00, 900.00, 120.00, 60.00],
    "quantidade": [1, 3, 2, 1, 2, 5, 1, np.nan, 4, 2],
    "data_venda": ["2025-01-05", "2025-01-06", "2025-01-06", "2025-01-07",
                   "2025-01-08", "2025-01-08", "2025-01-09", "2025-01-10",
                   "2025-01-10", "2025-01-11"],
}

df = pd.DataFrame(dados_brutos)

print("=" * 60)
print("DADOS BRUTOS (com valores faltantes propositalmente)")
print("=" * 60)
print(df)
print("\nInformações gerais do DataFrame:")
print(df.info())



# 2. TRANSFORM — limpeza e enriquecimento dos dados


# 2.1 Verificar valores nulos
print("\nValores nulos por coluna:")
print(df.isnull().sum())

# 2.2 Tratar valores nulos
# - Linha sem nome de produto: não dá pra "adivinhar", então removemos
df = df.dropna(subset=["produto"])

# - Preço faltante: preenchemos com a média de preço daquele produto
df["preco_unitario"] = df.groupby("produto")["preco_unitario"] \
                          .transform(lambda x: x.fillna(x.mean()))

# - Quantidade faltante: assumimos 1 unidade como valor conservador
df["quantidade"] = df["quantidade"].fillna(1)

# 2.3 Corrigir tipos de dados
df["data_venda"] = pd.to_datetime(df["data_venda"])
df["quantidade"] = df["quantidade"].astype(int)

# 2.4 Criar coluna nova: valor total da venda
df["valor_total"] = df["preco_unitario"] * df["quantidade"]

print("\n" + "=" * 60)
print("DADOS TRATADOS")
print("=" * 60)
print(df)



# 3. ANÁLISE — respondendo perguntas de negócio com groupby


print("\n" + "=" * 60)
print("ANÁLISES")
print("=" * 60)

# Faturamento total por categoria
faturamento_categoria = df.groupby("categoria")["valor_total"] \
                           .sum() \
                           .sort_values(ascending=False)
print("\nFaturamento total por categoria:")
print(faturamento_categoria)

# Produto mais vendido (em quantidade)
qtd_por_produto = df.groupby("produto")["quantidade"].sum() \
                     .sort_values(ascending=False)
print("\nQuantidade vendida por produto:")
print(qtd_por_produto)

# Ticket médio geral
ticket_medio = df["valor_total"].mean()
print(f"\nTicket médio das vendas: R$ {ticket_medio:.2f}")



# 4. LOAD — salvando o resultado tratado
df.to_csv("vendas_tratadas.csv", index=False)
faturamento_categoria.to_csv("faturamento_por_categoria.csv")

print("\n✅ Pipeline concluído! Arquivos salvos:")
print("- vendas_tratadas.csv")
print("- faturamento_por_categoria.csv")

DADOS BRUTOS (com valores faltantes propositalmente)
   id_venda   produto    categoria  preco_unitario  quantidade  data_venda
0         1  Notebook  Eletrônicos          3500.0         1.0  2025-01-05
1         2     Mouse   Acessórios            50.0         3.0  2025-01-06
2         3   Teclado   Acessórios           120.0         2.0  2025-01-06
3         4   Monitor  Eletrônicos           900.0         1.0  2025-01-07
4         5  Notebook  Eletrônicos          3500.0         2.0  2025-01-08
5         6     Mouse   Acessórios             NaN         5.0  2025-01-08
6         7   Cadeira       Móveis           450.0         1.0  2025-01-09
7         8   Monitor  Eletrônicos           900.0         NaN  2025-01-10
8         9   Teclado   Acessórios           120.0         4.0  2025-01-10
9        10       NaN   Acessórios            60.0         2.0  2025-01-11

Informações gerais do DataFrame:
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 6 columns)